# 02 — Lunar limb: circular Hough transform

Two‑pass CHT (paper Section 3.2, step 1) on the RGB‑summed image of every
frame → `products/moon_centers.csv` (same columns as the legacy sheet).

Legacy source: `automated_sheet_for_centers_radius_and_SB.py`.

⏱ ~30 s per frame at full resolution (≈ 75 min for all 145 frames).
Set `LIMIT` to a small number to smoke‑test.  The table that produced the paper
is kept in `data/sheet_for_centers_and_radius_and_SB.csv`; `config.MOON_CENTERS_SOURCE`
decides which one steps 03/04 use.

In [ ]:
import sys, time
sys.path.insert(0, "..")          # config.py / utils.py live one level up
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from astropy.io import fits

import config, utils
%matplotlib inline

In [ ]:
LIMIT = None          # e.g. 3 for a quick test
frames = utils.list_calibrated_frames()[:LIMIT]
print(len(frames), "frames")

In [ ]:
from tqdm.auto import tqdm
rows = []
for f in tqdm(frames):
    rgb = fits.getdata(config.CALIBRATED_LIGHTS_DIR / f)
    gray = np.sum(rgb, axis=0)
    del rgb
    xc, yc, r = utils.find_moon_center_radius(gray)
    rows.append(dict(Filename=f, Exposure=utils.read_header(f)["EXPTIME"],
                     Moon_XC=xc, Moon_YC=yc, Moon_Radius=r, Mean_SB=float(np.mean(gray))))
moon = pd.DataFrame(rows)
moon.to_csv(config.MOON_CENTERS_CSV, index=False)
moon.head()

In [ ]:
# Compare with the legacy CHT run (first match per file, as the legacy code did)
leg = pd.read_csv(config.LEGACY_MOON_CENTERS_CSV)
leg["Filename"] = leg.Filename.str.split("/").str[-1]
leg = leg.drop_duplicates("Filename").set_index("Filename")
cmp = moon.set_index("Filename").join(leg, rsuffix="_legacy", how="inner")
for c in ("Moon_XC", "Moon_YC", "Moon_Radius"):
    d = (cmp[c] - cmp[c + "_legacy"]).abs()
    print(f"{c:12s} max|diff| = {d.max():.1f} px   frames differing: {(d > 0).sum()}/{len(d)}")

In [ ]:
# QA: overlay the fitted circle on a few frames
show = moon.sample(min(6, len(moon)), random_state=0)
fig, axes = plt.subplots(1, len(show), figsize=(3.2*len(show), 3))
for ax, (_, r) in zip(np.atleast_1d(axes), show.iterrows()):
    img = fits.getdata(config.CALIBRATED_LIGHTS_DIR / r.Filename, memmap=True)
    ax.imshow(utils.asinh_stretch(np.sum(np.asarray(img[:, ::8, ::8], float), 0)), cmap="gray", extent=(0, config.IMAGE_SHAPE[1], config.IMAGE_SHAPE[0], 0))
    ax.add_patch(plt.Circle((r.Moon_XC, r.Moon_YC), r.Moon_Radius, ec="deepskyblue", fc="none", lw=0.8))
    ax.set_title(r.Filename.replace("-cal.fits", ""), fontsize=8); ax.axis("off")
plt.tight_layout()